This code runs the Conformal prediction analysis for Glioma radiomics MRI features. 

We implement this pipeline in the following steps:
1) the modeling table that was created from preprocessing is partitioned based on the source, i.e., we place the TCGA and UCSF in Development cohort, and EGD and UTSW-GLIOMA in their respective external cohorts
2) within the development cohort, we train the RandomForest clasifier using pre-set hyperparameters with Platt Calibration.. and 5-fold cross validation. Within each fold, we carry out within the training fold:
- missing value imputation with median
- variance filter on the raw scale
- z-scaling
- correlation filter
- MRMR feature selection, 
and then calculate the out-of-fold probabilities for each fold
3) again, within development cohort, we train the full model same steps but using the features from 5-fold CV. We will be using this model ONLY for external evaluation on EGD and UTSW in next step. 
4) EVALUATION 1: we first calculate the AUC, AUPRC, Sensitivity, Specificity, PPV, NPV for 3 predictions:
- in dev using oof probabilities from fold-specific model
- in ext 1 using egd probabilities predicted from full dev model
- in ext 2 using utsw probabiltiies predicted form full dev model, 
and all of these also calculate the 95%CIs from 1000 iteration
5) EVALUATION 2: MONDRIAN CONFORMAL PREDICTION results - 200 random splits using 50% calibration and 50% testing (4 settings 80%, 85%, 90%, 95% target)
6) EVALUATION 3: COMPARATIVE ANALYSIS OF DIFFERENT DECISION RULES - since the reviewer requested for probability abstention as well, we prepare a view that looks at :
- Recalibrated Argmax - using the calibration split to fit platt sigmoid 
- Class-weighted argmax - - using class weights from calibration split
- probability sets - include subtypes from most to least until combined probability sums 90%
- probability abstention - we keep the argmax label only if its cutoff is above a threshold, else this method would abstain and basically return all 3 as the predicted set. This cut-off is chosen baed on what we see in mondrian CP, so at the same deferral rate
- single threshold conformal 
- class-conformal mondrian, 
and we record the coverage for each class, average set sizes, single predictions, and accuracy of single predictions


Setup

In [1]:
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from itertools import product
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, average_precision_score, balanced_accuracy_score
from mrmr import mrmr_classif
from crepes import ConformalClassifier
from importlib.metadata import version

print('crepes', version('crepes'))
print('mrmr-selection', version('mrmr-selection'))
print('scikit-learn', version('scikit-learn'))

pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 30)

OUT = 'core_outputs'
os.makedirs(OUT, exist_ok=True)

crepes 0.9.0
mrmr-selection 0.2.8
scikit-learn 1.8.0


1) the modeling table that was created from preprocessing is partitioned based on the source, i.e., we place the TCGA and UCSF in Development cohort, and EGD and UTSW-GLIOMA in their respective external cohorts

In [2]:
table = pd.read_csv('modeling_table.csv')
print("\nThe full dataset is: ", str(table.shape))
print("\nUnique sites: ", table['site'].unique())
print("\nUnique scanners: ", table['Scanner'].unique())
print("\nUnique glioma classes: ", table['2021_glioma_class'].unique())
print("\nNumber of patients in each site:")
print(table['site'].value_counts())

feature_cols = [c for c in table.columns if c not in ['Scanner', '2021_glioma_class', 'age', 'sex', 'case_id', 'site']]

df_dev = table[table['site'].isin(['TCGA', 'UCSF'])]
df_egd = table[table['site'] == 'EGD']
df_utsw = table[table['site'] == 'UTSW']

y_dev = df_dev['2021_glioma_class'].values
y_egd = df_egd['2021_glioma_class'].values
y_utsw = df_utsw['2021_glioma_class'].values
classes = np.unique(y_dev)          # astrocytoma, glioblastoma, oligodendroglioma (column order of every probability array)
print("\nClasses: ", classes)

# Preprocessing (whole dataset)
X_dev = df_dev[feature_cols].replace([np.inf, -np.inf], np.nan)
X_egd = df_egd[feature_cols].replace([np.inf, -np.inf], np.nan)
X_utsw = df_utsw[feature_cols].replace([np.inf, -np.inf], np.nan)

# Defining the RF model (fixed settings, not tuned).. Platt scaling with 5 inner folds. 
rf = RandomForestClassifier(n_estimators=500, class_weight='balanced_subsample', max_features='sqrt', random_state=42, n_jobs=-1)
rf_platt = CalibratedClassifierCV(estimator=rf, method='sigmoid', cv=5)




The full dataset is:  (1544, 4230)

Unique sites:  ['UTSW' 'EGD' 'TCGA' 'UCSF']

Unique scanners:  ['Siemens' 'GE' 'Philips' 'Other' 'Unknown']

Unique glioma classes:  ['glioblastoma' 'astrocytoma' 'oligodendroglioma']

Number of patients in each site:
site
UTSW    529
UCSF    471
EGD     404
TCGA    140
Name: count, dtype: int64

Classes:  ['astrocytoma' 'glioblastoma' 'oligodendroglioma']


Development: 5-fold cross-validation Every step (imputation, filters, scaling, mRMR, model) is fitted on the 4 training folds only, then applied unchanged to the held-out fold.

In [3]:
# oof_probas: out-of-fold probabilities for each fold
oof_probas = np.zeros((len(y_dev), 3))

# 5-fold cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (tr, va) in enumerate(skf.split(X_dev, y_dev)):
    # imputation
    imp = SimpleImputer(strategy='median')
    X_tr = pd.DataFrame(imp.fit_transform(X_dev.iloc[tr]), columns=feature_cols)
    X_va = pd.DataFrame(imp.transform(X_dev.iloc[va]), columns=feature_cols)  

    # variance filter, raw scale
    keep = X_tr.columns[X_tr.var() > 1e-5]
    scaler = StandardScaler()
    X_tr = pd.DataFrame(scaler.fit_transform(X_tr[keep]), columns=keep)
    X_va = pd.DataFrame(scaler.transform(X_va[keep]), columns=keep)

    # correlation filter
    upper = X_tr.corr().abs().where(np.triu(np.ones((X_tr.shape[1], X_tr.shape[1])), k=1).astype(bool))
    X_tr = X_tr.drop(columns=[c for c in upper.columns if any(upper[c] > 0.85)])   # drop one of each pair |r| > 0.85

    features = mrmr_classif(X=X_tr, y=pd.Series(y_dev[tr]), K=20, show_progress=False)
    model = clone(rf_platt)
    model.fit(X_tr[features].values, y_dev[tr])
    oof_probas[va] = model.predict_proba(X_va[features].values)

    print('fold', fold + 1, 'done')

np.save(os.path.join(OUT, 'oof_probas.npy'), oof_probas)



fold 1 done
fold 2 done
fold 3 done
fold 4 done
fold 5 done


Final model: the same steps once on all 611 development patients. Frozen. EGD and UTSW go through the development imputer, filters, scaler and model. Nothing is refitted on them.

In [4]:
imp = SimpleImputer(strategy='median')
X_tr = pd.DataFrame(imp.fit_transform(X_dev), columns=feature_cols)
X_e = pd.DataFrame(imp.transform(X_egd), columns=feature_cols)
X_u = pd.DataFrame(imp.transform(X_utsw), columns=feature_cols)


keep = X_tr.columns[X_tr.var() > 1e-5]
scaler = StandardScaler()
X_tr = pd.DataFrame(scaler.fit_transform(X_tr[keep]), columns=keep)
X_e = pd.DataFrame(scaler.transform(X_e[keep]), columns=keep)
X_u = pd.DataFrame(scaler.transform(X_u[keep]), columns=keep)

upper = X_tr.corr().abs().where(np.triu(np.ones((X_tr.shape[1], X_tr.shape[1])), k=1).astype(bool))
X_tr = X_tr.drop(columns=[c for c in upper.columns if any(upper[c] > 0.85)])

final_features = mrmr_classif(X=X_tr, y=pd.Series(y_dev), K=20, show_progress=False)
final_model = clone(rf_platt)
final_model.fit(X_tr[final_features], y_dev)
egd_probas = final_model.predict_proba(X_e[final_features])
utsw_probas = final_model.predict_proba(X_u[final_features])

np.save(os.path.join(OUT, 'egd_probas.npy'), egd_probas)
np.save(os.path.join(OUT, 'utsw_probas.npy'), utsw_probas)
print('final features:', final_features)

final features: ['T1CE_original_firstorder_90Percentile', 'T1_wavelet-LLL_glszm_ZoneEntropy', 'T2_log-sigma-2-0-mm-3D_glszm_GrayLevelNonUniformityNormalized', 'T1CE_wavelet-LLH_glszm_ZoneEntropy', 'T1CE_original_firstorder_10Percentile', 'T1CE_log-sigma-1-0-mm-3D_glcm_Imc2', 'T2_original_firstorder_10Percentile', 'T1CE_wavelet-LHL_glszm_ZoneEntropy', 'T1CE_log-sigma-1-0-mm-3D_glszm_GrayLevelNonUniformityNormalized', 'T1_original_firstorder_10Percentile', 'T1CE_wavelet-LLH_glszm_GrayLevelNonUniformityNormalized', 'T1CE_original_firstorder_Skewness', 'T1CE_wavelet-HLL_glszm_GrayLevelNonUniformityNormalized', 'T1CE_log-sigma-2-0-mm-3D_firstorder_Maximum', 'T1CE_wavelet-LHH_glszm_ZoneEntropy', 'T1CE_wavelet-LHL_glszm_GrayLevelNonUniformityNormalized', 'T1CE_wavelet-LLH_glcm_Imc2', 'T1_original_firstorder_RootMeanSquared', 'T1CE_original_firstorder_Maximum', 'T1CE_wavelet-HLL_glszm_ZoneEntropy']


3. Base classifier performance, per cohort and subtype (one subtype against the rest) Point estimates are computed on all patients of the cohort. The 95% CIs are the 2.5th and 97.5th percentiles of the bootstrap values. AUC and AUPRC use the probabilities. Sens, spec, PPV, NPV use the argmax label. 1000 bootstrap resamples of patients, seed 42. The resamples are drawn in the order dev, EGD, UTSW.

In [5]:
np.random.seed(42)
boot_dev = np.random.randint(0, len(y_dev), (1000, len(y_dev)))
boot_egd = np.random.randint(0, len(y_egd), (1000, len(y_egd)))
boot_utsw = np.random.randint(0, len(y_utsw), (1000, len(y_utsw)))

perf_rows = []

for cohort, y, p, boot in [('Dev OOF', y_dev, oof_probas, boot_dev), ('EGD', y_egd, egd_probas, boot_egd), ('UTSW', y_utsw, utsw_probas, boot_utsw)]:
    pred = classes[np.argmax(p, axis=1)]
    for c, cls in enumerate(classes):
        stats = np.full((1000, 6), np.nan)          # columns: AUC, AUPRC, sens, spec, PPV, NPV. NaN when undefined in a resample
        for b in range(1000):
            truth = y[boot[b]] == cls
            called = pred[boot[b]] == cls
            score = p[boot[b], c]
            if truth.any() and not truth.all():
                stats[b, 0] = roc_auc_score(truth, score)
                stats[b, 1] = average_precision_score(truth, score)
            tp = np.sum(called & truth)
            fp = np.sum(called & ~truth)
            tn = np.sum(~called & ~truth)
            fn = np.sum(~called & truth)
            stats[b, 2] = tp / (tp + fn) if (tp + fn) > 0 else np.nan
            stats[b, 3] = tn / (tn + fp) if (tn + fp) > 0 else np.nan
            stats[b, 4] = tp / (tp + fp) if (tp + fp) > 0 else np.nan
            stats[b, 5] = tn / (tn + fn) if (tn + fn) > 0 else np.nan

        # point estimate on all patients of the cohort
        truth = y == cls
        called = pred == cls
        tp = np.sum(called & truth)
        fp = np.sum(called & ~truth)
        tn = np.sum(~called & ~truth)
        fn = np.sum(~called & truth)
        point = [roc_auc_score(truth, p[:, c]),
                 average_precision_score(truth, p[:, c]),
                 tp / (tp + fn),
                 tn / (tn + fp),
                 tp / (tp + fp) if (tp + fp) > 0 else np.nan,
                 tn / (tn + fn)]

        row = {'cohort': cohort, 'class': cls}
        for m, name in enumerate(['AUC', 'AUPRC', 'Sens', 'Spec', 'PPV', 'NPV']):
            row[name] = point[m]
            row[name + '_lo'] = np.nanpercentile(stats[:, m], 2.5)
            row[name + '_hi'] = np.nanpercentile(stats[:, m], 97.5)
        perf_rows.append(row)

performance = pd.DataFrame(perf_rows)
performance.to_csv(os.path.join(OUT, 'performance.csv'), index=False)
print(performance[['cohort', 'class', 'AUC', 'AUPRC', 'Sens', 'Spec', 'PPV', 'NPV']].round(2))

    cohort              class   AUC  AUPRC  Sens  Spec   PPV   NPV
0  Dev OOF        astrocytoma  0.95   0.88  0.79  0.95  0.81  0.94
1  Dev OOF       glioblastoma  0.96   0.98  0.98  0.79  0.93  0.93
2  Dev OOF  oligodendroglioma  0.88   0.34  0.12  1.00  0.60  0.96
3      EGD        astrocytoma  0.92   0.67  0.79  0.86  0.58  0.95
4      EGD       glioblastoma  0.96   0.98  0.96  0.71  0.85  0.91
5      EGD  oligodendroglioma  0.87   0.52  0.10  0.99  0.70  0.83
6     UTSW        astrocytoma  0.90   0.60  0.70  0.92  0.59  0.95
7     UTSW       glioblastoma  0.91   0.96  0.95  0.58  0.86  0.80
8     UTSW  oligodendroglioma  0.82   0.36  0.06  0.99  0.44  0.89


4. Mondrian conformal prediction, 200 splits per external cohort Each split: half the cohort (stratified by subtype) is the calibration half, the other half is the test half.
Score of a subtype = 1 - its probability. Mondrian: bins=y keeps a separate list of calibration scores for each true subtype, so each subtype gets its own cutoff. A test patient's set holds every subtype whose p-value is at least 1 - target.


In [6]:
targets = [0.80, 0.85, 0.90, 0.95]
cohorts = {'EGD': (y_egd, egd_probas), 'UTSW': (y_utsw, utsw_probas)}
is_test = {'EGD': np.zeros((200, len(y_egd)), dtype=bool), 'UTSW': np.zeros((200, len(y_utsw)), dtype=bool)}

cp_rows = []
for cohort, seed in product(['EGD', 'UTSW'], range(200)):
    y, p = cohorts[cohort]
    scores = 1 - p
    cal, test = train_test_split(np.arange(len(y)), test_size=0.5, random_state=seed, stratify=y)
    is_test[cohort][seed, test] = True

    true_col = np.searchsorted(classes, y)                     # column of each patient's true subtype
    cc = ConformalClassifier()
    cc.fit(scores[cal, true_col[cal]], bins=y[cal])           # calibration scores of the true subtype, grouped by subtype

    for target in targets:
        # one column per candidate subtype: put every test patient in that subtype's bin, keep that subtype's column
        sets = np.column_stack([cc.predict_set(scores[test], bins=np.full(len(test), cls), confidence=target, smoothing=False)[:, c]
                                for c, cls in enumerate(classes)])
        split = pd.DataFrame({'class': y[test], 'covered': sets[np.arange(len(test)), true_col[test]], 'size': sets.sum(axis=1)})
        split = split.groupby('class').mean().reset_index()
        split['cohort'] = cohort
        split['seed'] = seed
        split['target'] = target
        cp_rows.append(split)

cp = pd.concat(cp_rows, ignore_index=True)
cp.to_csv(os.path.join(OUT, 'conformal_200_splits.csv'), index=False)
np.save(os.path.join(OUT, 'is_test_egd.npy'), is_test['EGD'])
np.save(os.path.join(OUT, 'is_test_utsw.npy'), is_test['UTSW'])

# summary across the 200 splits: mean and 2.5th / 97.5th percentiles
cp_summary = cp.groupby(['cohort', 'target', 'class'])[['covered', 'size']].describe(percentiles=[0.025, 0.975])
print(cp_summary.round(2))


                                covered                                             size                                          
                                  count  mean   std   min  2.5%   50% 97.5%   max  count  mean   std   min  2.5%   50% 97.5%   max
cohort target class                                                                                                               
EGD    0.80   astrocytoma         200.0  0.83  0.08  0.59  0.67  0.84  0.95  1.00  200.0  1.36  0.11  1.00  1.13  1.37  1.55  1.59
              glioblastoma        200.0  0.81  0.04  0.65  0.72  0.81  0.89  0.92  200.0  0.93  0.04  0.83  0.87  0.94  1.00  1.04
              oligodendroglioma   200.0  0.80  0.09  0.59  0.62  0.81  0.95  0.97  200.0  1.36  0.11  0.97  1.13  1.36  1.57  1.59
       0.85   astrocytoma         200.0  0.87  0.08  0.63  0.71  0.88  0.97  1.00  200.0  1.52  0.15  1.16  1.26  1.50  1.82  1.87
              glioblastoma        200.0  0.85  0.04  0.72  0.76  0.85  0.93  0.96  

In [7]:
level = 0.90
print(level)

eye3 = np.eye(3, dtype=int)
print(eye3)

weight_grid = [np.array(w, dtype=float) for w in product([0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0], repeat=3)]
print(weight_grid)

rule_rows = []
for cohort, seed in product(['EGD', 'UTSW'], range(200)):
    y, p = cohorts[cohort]
    true_col = np.searchsorted(classes, y)
    test = np.where(is_test[cohort][seed])[0]
    cal = np.where(~is_test[cohort][seed])[0]
    scores = 1 - p
    p_clip = np.clip(p, 1e-6, 1 - 1e-6)
    log_odds = np.log(p_clip / (1 - p_clip))

    # 1. recalibration: one Platt sigmoid per subtype (C=1e6 means no penalty), fitted on the calibration half
    recal = np.zeros(p.shape)
    for k in range(3):
        sig = LogisticRegression(C=1e6, max_iter=1000).fit(log_odds[cal, k].reshape(-1, 1), (true_col[cal] == k).astype(int))
        recal[:, k] = sig.predict_proba(log_odds[:, k].reshape(-1, 1))[:, 1]
    recal = recal / recal.sum(axis=1, keepdims=True)
    set_recal_argmax = eye3[np.argmax(recal[test], axis=1)]

    # 2. class weights: first weight vector with the best balanced accuracy on the calibration half
    bal = [balanced_accuracy_score(true_col[cal], np.argmax(p[cal] * w, axis=1)) for w in weight_grid]
    best_w = weight_grid[int(np.argmax(bal))]
    set_weighted = eye3[np.argmax(p[test] * best_w, axis=1)]

    # 3. probability sets: keep a subtype if the probability of the more likely subtypes before it is still below 90%
    order = np.argsort(-recal[test], axis=1)
    sorted_p = np.take_along_axis(recal[test], order, axis=1)
    set_prob = np.zeros((len(test), 3), dtype=int)
    np.put_along_axis(set_prob, order, ((np.cumsum(sorted_p, axis=1) - sorted_p) < level).astype(int), axis=1)

    # 5 and 6. conformal, fitted on the calibration half
    cc_m = ConformalClassifier()
    cc_m.fit(scores[cal, true_col[cal]], bins=y[cal])
    cc_1 = ConformalClassifier()
    cc_1.fit(scores[cal, true_col[cal]])
    set_mondrian = np.column_stack([cc_m.predict_set(scores[test], bins=np.full(len(test), cls), confidence=level, smoothing=False)[:, c]
                                    for c, cls in enumerate(classes)])
    set_single = cc_1.predict_set(scores[test], confidence=level, smoothing=False)

    # 4. abstention: commit share = Mondrian single-label share on the calibration half
    cal_mondrian = np.column_stack([cc_m.predict_set(scores[cal], bins=np.full(len(cal), cls), confidence=level, smoothing=False)[:, c]
                                    for c, cls in enumerate(classes)])
    n_commit = int(round(np.mean(cal_mondrian.sum(axis=1) == 1) * len(cal)))
    cutoff = np.sort(p[cal].max(axis=1))[::-1][n_commit - 1] if n_commit > 0 else 1.01   # top p of the n_commit-th most confident
    commit = p[test].max(axis=1) >= cutoff
    set_abstain = np.ones((len(test), 3), dtype=int)
    set_abstain[commit] = eye3[np.argmax(p[test], axis=1)][commit]

    # score every rule on the test half
    yt = true_col[test]
    for rule, s in [('Recalibrated argmax', set_recal_argmax), ('Locally weighted argmax', set_weighted),
                    ('Recalibrated probability sets', set_prob), ('Probability abstention', set_abstain),
                    ('Single-threshold conformal', set_single), ('Mondrian conformal', set_mondrian)]:
        covered = s[np.arange(len(test)), yt] == 1
        size = s.sum(axis=1)
        single = size == 1
        rule_rows.append({'cohort': cohort, 'seed': seed, 'rule': rule,
                          'cov_A': covered[yt == 0].mean(), 'cov_G': covered[yt == 1].mean(), 'cov_O': covered[yt == 2].mean(),
                          'size': size.mean(), 'O_listed_for_nonO': s[yt != 2, 2].mean(),
                          'single_pct': single.mean() * 100,
                          'acc_single_pct': covered[single].mean() * 100 if single.any() else np.nan})

rules = pd.DataFrame(rule_rows)
rules.to_csv(os.path.join(OUT, 'decision_rules_200_splits.csv'), index=False)

# Table 1 numbers: mean over the 200 test halves
print(rules.groupby(['cohort', 'rule'], sort=False).mean(numeric_only=True).drop(columns='seed').round(2))


0.9
[[1 0 0]
 [0 1 0]
 [0 0 1]]
[array([0.5, 0.5, 0.5]), array([0.5 , 0.5 , 0.75]), array([0.5, 0.5, 1. ]), array([0.5, 0.5, 1.5]), array([0.5, 0.5, 2. ]), array([0.5, 0.5, 3. ]), array([0.5, 0.5, 4. ]), array([0.5, 0.5, 6. ]), array([0.5, 0.5, 8. ]), array([0.5 , 0.75, 0.5 ]), array([0.5 , 0.75, 0.75]), array([0.5 , 0.75, 1.  ]), array([0.5 , 0.75, 1.5 ]), array([0.5 , 0.75, 2.  ]), array([0.5 , 0.75, 3.  ]), array([0.5 , 0.75, 4.  ]), array([0.5 , 0.75, 6.  ]), array([0.5 , 0.75, 8.  ]), array([0.5, 1. , 0.5]), array([0.5 , 1.  , 0.75]), array([0.5, 1. , 1. ]), array([0.5, 1. , 1.5]), array([0.5, 1. , 2. ]), array([0.5, 1. , 3. ]), array([0.5, 1. , 4. ]), array([0.5, 1. , 6. ]), array([0.5, 1. , 8. ]), array([0.5, 1.5, 0.5]), array([0.5 , 1.5 , 0.75]), array([0.5, 1.5, 1. ]), array([0.5, 1.5, 1.5]), array([0.5, 1.5, 2. ]), array([0.5, 1.5, 3. ]), array([0.5, 1.5, 4. ]), array([0.5, 1.5, 6. ]), array([0.5, 1.5, 8. ]), array([0.5, 2. , 0.5]), array([0.5 , 2.  , 0.75]), array([0.5, 2. ,